# Проект II: Оценка пространственной доступности

В этом задании вы оцените **пространственную доступность** двух типов объектов для жителей выбранного города. Город и категории объектов предопределны заранее, вам нужно выбрать и записаться в [таблице](https://docs.google.com/spreadsheets/d/1rP5WBc4iVrE3erflx86WCk5sQAdgOvhILLhjfW1lrBM/edit?usp=sharing).

Ваша задача:

- построить зоны пешей доступности вокруг объектов каждой категории;
- используя данные о жилой площади МКД, оценить численность населения в каждом доме;
- определить, какая **доля жителей** попадает и не попадает в зону доступности для каждой из двух категорий.

В рамках задания мы:

- загрузим данные по двум назначенным категориям объектов и данные МКД;
- изучим, подготовим и перепроецируем данные;
- построим зоны пешей доступности (буфер или изохроны — на выбор);
- оценим численность населения по жилой площади;
- определим, какие дома и сколько жителей находятся вне зон доступности;
- визуализируем результаты.

Критерии оценивания (max. 10 баллов):

1. Корректность данных (2 балла)

- 1 балл – Данные по обеим категориям загружены без ошибок.
- 1 балл – Данные МКД прочитаны, на их основе создан GeoDataFrame

2. Расчёты и анализ (4 балла)

- 1 балл – Зоны доступности построены для обеих категорий.
- 1 балл – Оценочная численность населения рассчитана корректно.
- 1 балл – Для каждой категории определены здания вне зоны доступности.
- 1 балл – Рассчитана доля населения вне зоны доступности для каждой категории.

3. Визуализация (2 балла)

- 1 балл – Создана карта доступности для первой категории.
- 1 балл – Создана карта доступности для второй категории.

4. Оформление итогового результата (2 балла)

- 1 балл – Сданный код структурирован и читаем.
- 1 балл – Дана интерпретация результатов (Шаг 9).

> Работа сдаётся как документ в формате `.ipynb`


### Выбор города и категорий

**Таблица записи:** [открыть](https://docs.google.com/spreadsheets/d/1rP5WBc4iVrE3erflx86WCk5sQAdgOvhILLhjfW1lrBM/edit?usp=sharing)


In [ ]:
city_name = ""          # например, "Томск"
category_1_name = ""    # например, "Школы"
category_2_name = ""    # например, "Аптеки"

### Шаг 0. Импорт библиотек

Перед началом работы импортируйте все библиотеки, которые потребуются для выполнения задания.


In [ ]:
# ваш импорт библиотек

### Шаг 1. Загрузка данных

Вам нужны три набора данных:

1. **Объекты двух категорий** — скачайте самостоятельно из OSM (Вариант А) или используйте предподготовленные файлы (Вариант Б).
2. **Данные МКД** — прочитайте данные из csv и создайте на их основе GeoDataFrame — ссылка на файлы ниже в п. 1.5.
3. **Граница города**

Перед сдачей удалите неиспользуемые варианты загрузки.

---

**Вариант А — самостоятельная выгрузка из OSM по названию города**

#### 1.1. Определение территории


In [ ]:
# area_name = ""  # название города для geocode_to_gdf (например, "Томск, Томская область, Россия")

#### 1.2. Загрузка границы города


In [ ]:
# area_osm = ox.geocode_to_gdf(area_name)

#### 1.3. Теги для выгрузки из OSM

Ниже приведены теги для каждой из шести возможных категорий. Вы можете их дополнять и изменять при желании.

| Категория              | Теги OSM                                                               |
| ---------------------- | ---------------------------------------------------------------------- |
| Остановки ОТ           | `{"highway": "bus_stop", "railway": ["tram_stop", "station", "halt"]}` |
| Школы                  | `{"amenity": "school"}`                                                |
| Детские сады           | `{"amenity": "kindergarten"}`                                          |
| Медицинские учреждения | `{"amenity": ["hospital", "clinic"]}`                                  |
| Аптеки                 | `{"amenity": "pharmacy"}`                                              |
| Продуктовые магазины   | `{"shop": ["supermarket", "convenience", "grocery", "food"]}`          |


In [ ]:
# tags_cat1 = ...  # теги для первой категории
# tags_cat2 = ...  # теги для второй категории

#### 1.4. Загрузка данных из OSM


In [ ]:
# cat1 = ox.features_from_place(area_name, tags=tags_cat1)
# cat2 = ox.features_from_place(area_name, tags=tags_cat2)

---

**Вариант Б — предподготовленные данные OSM**

Для каждого города уже подготовлены GeoPackage-файлы со всеми шестью категориями объектов.

> **Данные OSM по городам:** [ссылка](https://drive.google.com/drive/folders/1_GGGUVlRtHuR1lTFj1PUiSuAurPnZw-o?usp=sharing)

Скачайте файл своего города, положите рядом с ноутбуком и прочитайте нужные слои.

Вспомните, как читать конкретный слой из GeoPackage и как посмотреть, какие слои в нём содержатся.


In [ ]:
# area_osm = gpd.read_file("НазваниеГорода.gpkg", layer="...")
# cat1 = gpd.read_file("НазваниеГорода.gpkg", layer="...")  # слой первой категории
# cat2 = gpd.read_file("НазваниеГорода.gpkg", layer="...")  # слой второй категории

#### 1.5. Загрузка данных МКД

Данные о жилой площади многоквартирных домов предоставлены для каждого города.

> **Данные МКД:** [открыть папку](https://drive.google.com/drive/folders/1WAr8ZtgCfWQXxx24pM6O-21s1R97N9It?usp=share_link)

Скачайте файл своего города, положите рядом с ноутбуком и прочитайте его. Проверьте, что информация прочиталась корректно.


In [ ]:
# mkd = ...  # прочитайте файл с данными МКД

Создайте на основе прочитанных вами данных точечный слой.


In [ ]:
# Проверьте, как называются колонки с координатами — например, 'lat'/'lon', 'x'/'y', 'longitude'/'latitude'
# print(mkd.columns.tolist())

# buildings = gpd.GeoDataFrame(mkd, geometry=gpd.points_from_xy(mkd['???'], mkd['???']), crs='EPSG:4326')

### Шаг 2. Изучение данных

Перед началом анализа важно понять, что именно содержится в загруженных наборах данных.

Самостоятельно определите, какая информация вам нужна, и изучите данные.


In [ ]:
# ваш код

### Шаг 3. Обработка данных

#### 3.1. Обработка геометрии

Для построения зон доступности нам нужны **точки** — центры объектов. Но OSM-данные часто содержат смешанную геометрию: одни объекты (например, маленький детский сад) отмечены точкой, другие (большой школьный комплекс) — полигоном.

**Один из вариатов:** оставить точки как есть, а из полигонов взять центроид (`centroid`). В итоге все объекты станут точками.


In [ ]:
# Приводим обе категории к точкам:
# centroid точки — это сама точка, centroid полигона — его центр
# cat1['geometry'] = cat1.geometry.centroid
# cat2['geometry'] = cat2.geometry.centroid

#### 3.2. Отбор атрибутов

Удалите лишние поля, оставьте только необходимые. Обязательно сохраните `geometry`.

Рекомендуется также сохранить `osmid` и поля, идентифицирующие тип объекта.


In [ ]:
# ваш код

### Шаг 4. Перепроецирование данных

Для построения буферных зон данные должны быть в **проецированной системе координат** (единицы — метры).

#### 4.1. Определение подходящей CRS


In [ ]:
# ваш код

#### 4.2. Перепроецирование

Перепроецируйте все наборы данных в выбранную систему координат.


In [ ]:
# ваш код
# buildings_utm = buildings.to_crs(...)  # МКД-здания в той же проецированной CRS

### Шаг 5. Построение зон доступности

Наше задание предполагает построение зон доступности в виде буферных зон, однако при желании вы также можете использовать изохроны, однако эту тему мы будем рассматривать подробнее в следующих темах нашего курса.

#### 5.1. Зоны доступности — первая категория (`cat1`)

Определите радиус или время ходьбы. Если нормативное значение известно — используйте его и укажите источник в интерпретации.

> **Опционально (доп. задание):** если ваша категория включает объекты разных типов (например, в «Медицинских учреждениях» — больницы и поликлиники, в «Остановках ОТ» — метро и автобусные остановки), можно задать **разные радиусы для каждого подтипа** и построить отдельные зоны. Если не хотите усложнять — стройте один радиус для всех объектов категории.


In [ ]:
# ваш код 

#### 5.2. Объединённая зона доступности — первая категория

Объедините зоны в единую с помощью `dissolve()`.


In [ ]:
# ваш код
# cat1_zone_union = cat1_zone.dissolve()

#### 5.3. Зоны доступности — вторая категория (`cat2`)

> **Опционально (доп. задание):** аналогично п. 5.1 — разные радиусы для разных подтипов.


In [ ]:
# ваш код

#### 5.4. Объединённая зона доступности — вторая категория


In [ ]:
# ваш код
# cat2_zone_union = cat2_zone.dissolve()

### Шаг 6. Оценка численности населения

Чтобы посчитать долю жителей вне зон доступности, нужно знать, сколько человек живёт в каждом доме.


#### 6.1. Расчёт оценочной численности населения

Рассчитайте оценочную численность населения в каждом здании:

$$\text{население} = \frac{\text{жилая площадь}}{\text{норма площади на человека / обеспеченность}}$$

Запишите выбранное значение обеспеченность в переменную `area_per_person` и обоснуйте его кратко.


In [ ]:
# area_per_person = 22  # м² на человека

# buildings_utm['population'] = buildings_utm['living_area'] / area_per_person

# total_population = buildings_utm['population'].sum()
# print(f"Оценочная численность населения: {total_population:.0f} чел.")

### Шаг 7. Оценка доступности населения

#### 7.1. Классификация зданий — первая категория

Определите, какие здания попадают в зону доступности первой категории и добавьте колонку `access_cat1` (`True` / `False`).

Это можно сделать через `.intersects()`.


In [ ]:
# Пример
# buildings_utm['access_cat1'] = buildings_utm.geometry.intersects(
#     cat1_zone_union.geometry.iloc[0]
# )


#### 7.2. Статистика доступности — первая категория

Рассчитайте суммарную оценочную численность населения, проживающую в зоне доступности и за её пределами для первой категории объектов, и определите доли каждой группы.


In [ ]:
# Пример
# pop_no_cat1 = buildings_utm.loc[buildings_utm['access_cat1'] == False, 'population'].sum()
# share_no_cat1 = pop_no_cat1 / total_population * 100
# print(f"Вне зоны доступности ({category_1_name}): {pop_no_cat1:.0f} чел. ({share_no_cat1:.1f}%)")

#### 7.3. Классификация зданий — вторая категория

Аналогично — добавьте колонку `access_cat2` (`True` / `False`).


In [ ]:
# Пример
# buildings_utm['access_cat2'] = buildings_utm.geometry.intersects(
#     cat2_zone_union.geometry.iloc[0]
# )


#### 7.4. Статистика доступности — вторая категория

Рассчитайте суммарную оценочную численность населения, проживающую в зоне доступности и за её пределами для второй категории объектов, и определите доли каждой группы.


In [ ]:
# Пример
# pop_no_cat2 = buildings_utm.loc[buildings_utm['access_cat2'] == False, 'population'].sum()
# share_no_cat2 = pop_no_cat2 / total_population * 100
# print(f"Вне зоны доступности ({category_2_name}): {pop_no_cat2:.0f} чел. ({share_no_cat2:.1f}%)")

### Шаг 8. Визуализация результатов


#### 8.1. Доступность — первая категория

Создайте карту, на которой здания окрашены по доступности:

- здания (точки МКД) в зоне доступности — одним цветом;
- здания вне зоны доступности — другим.
- отобразите зону доступности
- отобразите сами объекты (опционально)


In [ ]:
# ваш код

#### 8.2. Доступность — вторая категория

Аналогичная карта для второй категории.


In [ ]:
# ваш код

### Шаг 9. Интерпретация результатов

Кратко ответьте на следующие вопросы:

- Какова доля населения вне зоны доступности для каждой категории?
- Обоснуйте выбор радиуса для зон доступности: на что вы опирались?
- Как качество данных OSM и использование МКД может влиять на достоверность результатов?


### Шаг 10. Оценка времени на выполнение задания

Напишите, сколько примерно времени вы затратили.

(на оценку не влияет — нужно для статистики)


X часов
